In [1]:
from google.colab import drive
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dev-public.zip"
!unzip -q "$zip_path" -d /content/dataset
!ls /content/dataset

Mounted at /content/drive
test  train


In [2]:
import os, random, zipfile
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [3]:
class UNetSmall(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU()
        )
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )
        self.out = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d1 = self.up1(b)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.dec1(d1)
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        return self.out(d2)





In [4]:
def get_transform(size=(256, 256), train=True):
    if train:
        return T.Compose([
            T.Resize(size),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(0.2, 0.2, 0.2, 0.1),
            T.ToTensor(),
            T.Normalize(mean=[0.5]*3, std=[0.5]*3)
        ])
    else:
        return T.Compose([
            T.Resize(size),
            T.ToTensor(),
            T.Normalize(mean=[0.5]*3, std=[0.5]*3)
        ])

COLOR_TO_ID = {
    (0, 0, 0): 0, (255, 255, 0): 1, (0, 255, 255): 2, (204, 0, 0): 3,
    (255, 0, 0): 4, (204, 0, 204): 5, (76, 153, 0): 6, (0, 51, 0): 7,
    (102, 51, 0): 8, (204, 204, 0): 9, (0, 0, 153): 10, (255, 204, 204): 11,
    (255, 51, 153): 12, (0, 0, 204): 13, (102, 204, 0): 14, (255, 153, 51): 15,
    (0, 204, 0): 16, (51, 51, 255): 17, (0, 204, 204): 18
}

def mask_to_ids(mask_pil, size=(256,256), num_classes=19, ignore_index=255):
    mask_pil = mask_pil.resize(size, resample=Image.NEAREST)
    arr = np.array(mask_pil)
    out = np.ones(arr.shape[:2], dtype=np.int64) * ignore_index
    for c, i in COLOR_TO_ID.items():
        out[np.all(arr == c, axis=-1)] = i
    out[(out < 0) | (out >= num_classes)] = ignore_index
    return out


In [5]:
class FaceDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, cache_dir=None,
                 transform=None, num_classes=19, ignore_index=255, size=(256,256)):
        self.img_dir, self.mask_dir, self.transform = img_dir, mask_dir, transform
        self.num_classes, self.ignore_index, self.size = num_classes, ignore_index, size
        self.images = sorted(os.listdir(img_dir))
        self.cache_dir = cache_dir
        if self.mask_dir:
            os.makedirs(cache_dir, exist_ok=True)
            print(f"Preparing masks (cache: {cache_dir}) ...")
            for img in self.images:
                base = os.path.splitext(img)[0]
                cache_path = os.path.join(cache_dir, f"{base}.npy")
                if not os.path.exists(cache_path):
                    mpath = os.path.join(mask_dir, base + '.png')
                    mask = Image.open(mpath).convert("RGB")
                    mask_id = mask_to_ids(mask, size=self.size)
                    np.save(cache_path, mask_id)
            print("All masks prepared and cached.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]
        img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.mask_dir is None:
            return img
        mask = np.load(os.path.join(self.cache_dir, os.path.splitext(name)[0] + ".npy"))
        mask = torch.from_numpy(mask)
        mask[(mask < 0) | (mask >= self.num_classes)] = self.ignore_index
        return img, mask


In [6]:
NUM_CLASSES = 19
EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 8
IMG_SIZE = (256, 256)
IGNORE_INDEX = 255
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_img = "/content/dataset/train/images"
train_mask = "/content/dataset/train/masks"
test_img = "/content/dataset/test/images"
cache_dir = "/content/dataset/masks_prepared_256"

train_tf = get_transform(IMG_SIZE, True)
val_tf = get_transform(IMG_SIZE, False)

dataset = FaceDataset(train_img, train_mask, cache_dir, transform=None,
                      num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, size=IMG_SIZE)

n_total = len(dataset)
n_val = int(0.1 * n_total)
n_train = n_total - n_val
train_set, val_set = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

train_set.dataset.transform = train_tf
val_set.dataset.transform = val_tf

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training images: {len(train_set)}, Validation images: {len(val_set)}")

model = UNetSmall(NUM_CLASSES).to(DEVICE)
ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total trainable parameters: {num_params:,}")

def dice_loss(pred, target, num_classes=NUM_CLASSES, eps=1e-7):
    pred = torch.softmax(pred, dim=1)
    onehot = F.one_hot(target, num_classes).permute(0,3,1,2).float()
    inter = torch.sum(pred * onehot, (0,2,3))
    card = torch.sum(pred + onehot, (0,2,3))
    dice = (2 * inter + eps) / (card + eps)
    return 1 - dice.mean()

def combined_loss(logits, target):
    return ce_loss(logits, target) + dice_loss(logits, target)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


Preparing masks (cache: /content/dataset/masks_prepared_256) ...
All masks prepared and cached.
Training images: 900, Validation images: 100
Total trainable parameters: 467,123


In [7]:
@torch.no_grad()
def pixel_accuracy(model, loader, max_batches=None):
    model.eval()
    correct, total = 0, 0
    for i, (images, masks) in enumerate(loader):
        if max_batches and i >= max_batches: break
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        preds = model(images).argmax(dim=1)
        correct += (preds == masks).sum().item()
        total += masks.numel()
    return correct / total if total > 0 else 0

@torch.no_grad()
def dice_score(model, loader, num_classes=NUM_CLASSES, max_batches=None):
    model.eval()
    eps = 1e-7
    scores = []
    for i, (images, masks) in enumerate(loader):
        if max_batches and i >= max_batches: break
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        preds = model(images).argmax(dim=1)
        for b in range(preds.size(0)):
            f1_per_class = []
            for c in range(num_classes):
                p = (preds[b] == c)
                g = (masks[b] == c)
                tp = (p & g).sum().item()
                fp = (p & (~g)).sum().item()
                fn = ((~p) & g).sum().item()
                denom = (2*tp + fp + fn)
                if denom == 0: continue
                f1_per_class.append((2*tp)/(denom+eps))
            if f1_per_class:
                scores.append(np.mean(f1_per_class))
    return float(np.mean(scores)) if scores else 0.0

best_val_f1, best_path = -1.0, "model_best.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    acc = pixel_accuracy(model, val_loader, max_batches=5)
    val_f1 = dice_score(model, val_loader, NUM_CLASSES, max_batches=5)
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Acc: {acc*100:.2f}% | F1: {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_path)
        print(f"✅ New best model saved: {best_path} (F1={best_val_f1:.4f})")

print(f"Training finished. Best Val F1: {best_val_f1:.4f}")


Epoch 1/50 | Loss: 2.7324 | Acc: 52.47% | F1: 0.1108
✅ New best model saved: model_best.pth (F1=0.1108)
Epoch 2/50 | Loss: 2.2497 | Acc: 58.92% | F1: 0.1303
✅ New best model saved: model_best.pth (F1=0.1303)
Epoch 3/50 | Loss: 2.1442 | Acc: 59.52% | F1: 0.1322
✅ New best model saved: model_best.pth (F1=0.1322)
Epoch 4/50 | Loss: 2.0833 | Acc: 61.22% | F1: 0.1970
✅ New best model saved: model_best.pth (F1=0.1970)
Epoch 5/50 | Loss: 1.9878 | Acc: 61.89% | F1: 0.2176
✅ New best model saved: model_best.pth (F1=0.2176)
Epoch 6/50 | Loss: 1.9156 | Acc: 64.16% | F1: 0.2823
✅ New best model saved: model_best.pth (F1=0.2823)
Epoch 7/50 | Loss: 1.8222 | Acc: 64.42% | F1: 0.3283
✅ New best model saved: model_best.pth (F1=0.3283)
Epoch 8/50 | Loss: 1.7255 | Acc: 67.17% | F1: 0.3484
✅ New best model saved: model_best.pth (F1=0.3484)
Epoch 9/50 | Loss: 1.6658 | Acc: 66.33% | F1: 0.3839
✅ New best model saved: model_best.pth (F1=0.3839)
Epoch 10/50 | Loss: 1.5969 | Acc: 71.00% | F1: 0.4005
✅ New best

In [8]:
test_dir = "/content/dataset/test/images"
output_dir = "/content/submission/masks"
model_path = "model_best.pth"
os.makedirs(output_dir, exist_ok=True)

model = UNetSmall(19).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.eval()

test_tf = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

with torch.no_grad():
    for fname in tqdm(sorted(os.listdir(test_dir))):
        if not fname.lower().endswith((".png", ".jpg")):
            continue
        img_path = os.path.join(test_dir, fname)
        img = Image.open(img_path).convert("RGB")
        inp = test_tf(img).unsqueeze(0).to(DEVICE)
        pred = model(inp).argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        pred_img = Image.fromarray(pred).resize((512, 512), resample=Image.NEAREST).convert("L")
        base = os.path.splitext(fname)[0]
        pred_img.save(os.path.join(output_dir, base + ".png"))

print(f"✅ {len(os.listdir(output_dir))} masks saved in {output_dir}")

submission_zip = "/content/submission.zip"
with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk("/content/submission"):
        for f in files:
            fp = os.path.join(root, f)
            arcname = os.path.relpath(fp, "/content/submission")
            zipf.write(fp, arcname)

print(f"✅ submission.zip created: {submission_zip}")


100%|██████████| 100/100 [00:01<00:00, 52.54it/s]

✅ 100 masks saved in /content/submission/masks
✅ submission.zip created: /content/submission.zip


In [9]:
import torch
import numpy as np
from tqdm import tqdm

def evaluate_f1(model, loader, num_classes=19):
    model.eval()
    eps = 1e-7
    class_scores = []

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Evaluating"):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            preds = model(images).argmax(dim=1)

            for c in range(num_classes):
                p = (preds == c)
                g = (masks == c)
                tp = (p & g).sum().item()
                fp = (p & (~g)).sum().item()
                fn = ((~p) & g).sum().item()
                denom = (2 * tp + fp + fn)
                if denom == 0:
                    continue
                class_scores.append((2 * tp) / (denom + eps))

    mean_f1 = float(np.mean(class_scores)) if class_scores else 0.0
    print(f"🔹 Mean F1-Score (Dice): {mean_f1:.4f}")
    return mean_f1

# Beispiel-Aufruf:
model.load_state_dict(torch.load("model_best.pth", map_location=DEVICE))
f1_val = evaluate_f1(model, val_loader, num_classes=19)


Evaluating: 100%|██████████| 13/13 [00:01<00:00, 10.00it/s]

🔹 Mean F1-Score (Dice): 0.6449
